# Day 9 — Machines That Act 🎮

> **Mission briefing:** Everything the Lab has built so far *perceives* — it classifies penguins, clusters colors, reads handwriting, understands words. Today we build something new: an agent that *decides*. There are no labels and no answer key — only a world, actions, and **rewards**. You will train a brain the way you would train a puppy: a treat for every good move.

**Previously in the Lab:** yesterday you unlocked the 💬 **Language Lab** and made transformer models write stories, read moods, and fill in the blanks.

**Today you will:**
- Meet **reinforcement learning** — the loop of *act → get a reward → learn*
- Teach an agent to cross a frozen lake with a **Q-table** (64 numbers that become an instinct)
- Write the single line of math — the **Q-learning update** — that sits at the heart of it
- Watch a neural network (a **Deep Q-Network**) learn to balance a pole from nothing
- See why this is how AlphaGo won, how robots learn to walk, and (as **RLHF**) part of how ChatGPT was polished

**Today you unlock:** 🔓 **🎮 Game Master** — replay both agents you trained, live in your Studio.

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║ SETUP — run me first! (identical in every Lab notebook)           ║
# ╚═══════════════════════════════════════════════════════════════════╝
import os, sys, pathlib

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("SMOKE_TEST", "0") == "1"   # tiny fast run for automated tests

REPO_URL = "https://github.com/A-Halimi/ai-studio-internship.git"  # instructor: set once

if IN_COLAB:
    if not pathlib.Path("/content/ai-studio-internship").exists():
        !git clone -q {REPO_URL} /content/ai-studio-internship
    %cd /content/ai-studio-internship/notebooks/day09                    # ← ADAPT day folder
    %pip -q install "gymnasium[classic-control,toy-text]==1.3.0" stable-baselines3==2.9.0 "imageio[ffmpeg]==2.37.3" gradio==6.19.0   # ← ADAPT per-day installs

HERE = pathlib.Path.cwd()
REPO = HERE.parents[1]                       # .../notebooks/dayNN -> repo root
DATA_DIR = pathlib.Path(os.environ.get("COURSE_DATA_DIR", REPO / "data"))
SAMPLES_DIR = REPO / "data_samples"          # small datasets shipped with the repo
MODELS_DIR = REPO / "ai_studio" / "models"   # where your Studio modules unlock
for p in (REPO / "data", MODELS_DIR):
    p.mkdir(parents=True, exist_ok=True)

import random
import numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print(f"✅ Setup done | Colab: {IN_COLAB} | Smoke test: {SMOKE}")
print(f"   data: {DATA_DIR}")

In [ ]:
import torch
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"⚡ Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("🐢 No GPU found — running on CPU. Everything still works, just slower.")

In [ ]:
# ── CONFIG — the Lab's control panel. Tweak me later! ──────────────
EPISODES  = 300 if SMOKE else 3000     # Q-learning episodes on FrozenLake
TIMESTEPS = 3_000 if SMOKE else 60_000 # Deep-Q-Network training steps on CartPole
ALPHA = 0.8    # learning rate for the Q-table (how big a step we take)
GAMMA = 0.95   # discount: how much the agent values *future* rewards (patience)
print(f"EPISODES={EPISODES} | TIMESTEPS={TIMESTEPS} | ALPHA={ALPHA} | GAMMA={GAMMA}")

## 1 · The reinforcement-learning loop

Supervised learning (Days 2–8) needs an **answer key**: a photo *labeled* "cat", a review *labeled* "positive". Reinforcement learning throws the answer key away. Instead there is a loop:

> The **agent** looks at the current **state**, picks an **action**, and the **environment** replies with a **reward** and a new state. Repeat thousands of times. The agent's only goal: collect as much reward as possible over the long run.

```
        ┌──────────────────────────────────────────┐
        │                                          │
   ┌────▼────┐   action a    ┌───────────────┐     │
   │  AGENT  │ ────────────▶ │  ENVIRONMENT  │     │
   │ (brain) │               │   (a world)   │     │
   └─────────┘ ◀──────────── └───────────────┘     │
        │        reward r + new state s'           │
        └──────────────────────────────────────────┘
```

The agent is never *told* the right move. It has to **discover** which moves pay off, purely from the rewards. That is the whole idea — and it is remarkably general:

| Situation | The reward the agent chases |
|---|---|
| Playing a video game | the score going up |
| A robot learning to walk | distance covered without falling over |
| A self-driving car | staying in its lane and reaching the destination |
| Polishing ChatGPT (**RLHF**) | a human clicking 👍 instead of 👎 |

One loop, a thousand applications. Let's build the smallest possible version of it.

## 2 · Meet FrozenLake ❄️

Our first world is a tiny 4×4 frozen pond from the **Gymnasium** library (the standard toolbox of RL practice environments). The rules:

- The agent starts at **S** (top-left) and wants the **gift** 🎁 at **G** (bottom-right).
- Some tiles are **holes** (H) — step on one and the episode ends with nothing.
- **16 states** (the tiles, numbered 0–15) and **4 actions**: `0=LEFT  1=DOWN  2=RIGHT  3=UP`.
- **Reward = 1** *only* when it reaches the gift. Every other step gives **0**.

That last point is what makes this hard: the agent gets almost no feedback. It has to stumble onto the gift by luck at least once, then learn to repeat the path. We use `is_slippery=False` so the ice behaves predictably — a move you choose is the move you make.

In [ ]:
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # let pygame draw with no screen (cloud/headless)
import gymnasium as gym
import matplotlib.pyplot as plt

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False,
               render_mode="rgb_array")
state, _ = env.reset(seed=SEED)

print("Number of states :", env.observation_space.n)   # 16 tiles
print("Number of actions:", env.action_space.n)        # 4 moves (L, D, R, U)
print("The pond (S=start  F=frozen  H=hole  G=gift):")
print(env.unwrapped.desc.astype(str))

plt.figure(figsize=(4, 4))
plt.imshow(env.render())
plt.axis("off")
plt.title("FrozenLake 4x4 — get from S (top-left) to the gift (bottom-right)")
plt.show()

### 🧪 Exercise 1 — how far does *pure luck* get?

Before we teach the agent anything, let's measure the baseline: an agent that ignores everything and picks **random** moves.

- Run many episodes where every action comes from `env.action_space.sample()`.
- Count how often it stumbles onto the gift, and report the success rate.

Expected outcome: a dismal **~1–5%**. Wandering blindly across ice rarely ends well.

In [ ]:
N_RANDOM = 200 if SMOKE else 1000
env.action_space.seed(SEED)          # make the "random" moves reproducible

wins = 0
for episode in range(N_RANDOM):
    state, _ = env.reset()
    done = False
    # ==================== YOUR CODE HERE ====================
    ### HINT: env.action_space.sample() returns a random action (0-3)
    ### HINT: after the episode, reward > 0 means it finished on the gift
    ...

success_rate = wins / N_RANDOM
print(f"🎲 Random agent reached the gift in {success_rate:.1%} of {N_RANDOM} episodes")

## 3 · The Q-table — the agent's cheat sheet

Here is the key idea. Imagine a table with **one row per state** and **one column per action**. Each cell holds a single number, `Q(state, action)` — the agent's estimate of *"how good is it to take this action from this state, if I play well afterward?"*

For FrozenLake that's a **16 × 4 = 64-number** table. If the agent could fill it in correctly, choosing the best move would be trivial: in any state, look at that row and pick the column with the biggest number.

The agent starts knowing **nothing**, so every cell begins at zero:

In [ ]:
q = np.zeros((16, 4))     # 16 states (rows) x 4 actions (columns)
print("The Q-table starts as all zeros — the agent knows nothing yet:")
print(q)
print("Shape:", q.shape, "→ one row per state, one column per action")

### 🧪 Exercise 2 — explore vs. exploit (`choose_action`)

An agent that always picks the highest number in its table will keep repeating whatever it tried first — it never discovers better moves. An agent that always moves randomly never *uses* what it learned. The fix is **ε-greedy**:

- With probability **ε** (epsilon): **explore** — take a random action.
- Otherwise: **exploit** — take the action with the largest Q-value for this state.

It's the restaurant dilemma: most nights you go to your favorite (exploit), but once in a while you try somewhere new (explore) in case it's even better. Fill in `choose_action`.

In [ ]:
def choose_action(state, q, eps):
    """Epsilon-greedy: explore with probability eps, otherwise exploit the Q-table."""
    # ==================== YOUR CODE HERE ====================
    ### HINT: np.random.random() draws a float in [0, 1)
    ### HINT: np.argmax(q[state]) is the column with the biggest Q-value for this state
    ...

# quick check: with eps=0 it must ALWAYS pick the greedy action
_probe = np.zeros((16, 4)); _probe[3] = [0, 0, 9, 0]
assert choose_action(3, _probe, eps=0.0) == 2, "With eps=0 you should always exploit!"
print("✅ choose_action works — with eps=0 it picks the best action (column 2 here).")

### 🧪 Exercise 3 — start bold, end wise (`get_epsilon`)

Early in training the agent should **explore a lot** (it knows nothing). Later it should mostly **exploit** what it has learned. So we let ε **decay** over the run — from `1.0` (all exploration) down to `0.01` (almost always greedy).

Write a **linear** schedule: on episode `0` return `EPS_START`; on the final episode return (about) `EPS_END`; blend smoothly in between.

In [ ]:
EPS_START, EPS_END = 1.0, 0.01
def get_epsilon(episode, n_episodes):
    """Linear decay from EPS_START (all exploration) to EPS_END (almost all exploit)."""
    # ==================== YOUR CODE HERE ====================
    ### HINT: frac should run from 0 (first episode) to 1 (last episode)
    ### HINT: blend EPS_START and EPS_END using frac
    ...

assert abs(get_epsilon(0, EPISODES) - 1.0) < 1e-9, "Episode 0 should be full exploration"
assert get_epsilon(EPISODES - 1, EPISODES) <= 0.02, "The last episode should be near-greedy"
print(f"✅ epsilon starts at {get_epsilon(0, EPISODES):.2f} "
      f"and ends at {get_epsilon(EPISODES-1, EPISODES):.2f}")

## 4 · ★ The Q-learning update — the heart of it all

Every time the agent takes action `a` in state `s`, lands in state `s'`, and collects reward `r`, it improves **one cell** of its table with this rule:

> **new opinion of (s, a)  =  old opinion  +  ALPHA × ( reward  +  GAMMA × best future value  −  old opinion )**

In code, that's a single line:

```python
q[s, a] += ALPHA * (r + GAMMA * np.max(q[s_next]) - q[s, a])
```

Read it slowly. `r + GAMMA * np.max(q[s_next])` is a **fresh estimate** of how good `(s, a)` really was — the reward we just got, plus the discounted value of the *best* move available from where we landed. We nudge the old value a fraction `ALPHA` of the way toward that fresh estimate. Do this a few thousand times and the table fills itself in.

<details><summary>🔢 <b>Math Corner</b> — the update in symbols (optional)</summary>

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\, r + \gamma \max_{a'} Q(s',a') - Q(s,a) \,\big]$$

- $Q(s,a)$ — how good we *currently* think action $a$ is in state $s$.
- $r$ — the reward we just received.
- $\max_{a'} Q(s',a')$ — the value of the **best** move from the new state $s'$.
- $\gamma$ (GAMMA) — a **patience knob** in $[0,1]$: near $0$ chases reward *now*; near $1$ plans far ahead.
- $\alpha$ (ALPHA) — the **learning rate**: the step size toward the new estimate.

The bracketed part is the **temporal-difference error** — the gap between what we expected and what happened. Learning is just shrinking that gap.

</details>

### 🧪 Exercise 4 — train the agent (the crown jewel) 👑

Everything comes together in one loop. For each episode we reset the pond, decay ε, then step until the episode ends — choosing an action and updating the table each step. **Two lines are yours to fill:**

1. Pick the action with the `choose_action` function you wrote.
2. Apply the **Q-learning update** from the section above.

Watch the printed success rate climb from near-zero toward **100%**.

In [ ]:
q = np.zeros((16, 4))          # start again from a blank cheat sheet
successes = []                  # 1.0 if the episode reached the gift, else 0.0

for episode in range(EPISODES):
    state, _ = env.reset()
    eps = get_epsilon(episode, EPISODES)
    done = False
    while not done:
        # ==================== YOUR CODE HERE ====================
        ### HINT: reuse the choose_action function you wrote in Exercise 2
        ...
        next_state, reward, terminated, truncated, _ = env.step(action)
        # ==================== YOUR CODE HERE ====================
        ### HINT: new = old + ALPHA * (reward + GAMMA * best_next - old)
        ...
        state = next_state
        done = terminated or truncated
    successes.append(1.0 if reward > 0 else 0.0)

    if (episode + 1) % max(1, EPISODES // 10) == 0:
        recent = np.mean(successes[-100:])
        print(f"  episode {episode+1:>5}/{EPISODES} | eps={eps:4.2f} | recent success = {recent:5.1%}")

print("🏁 Training done!")

In [ ]:
window = 100
rolling = np.convolve(successes, np.ones(window) / window, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(range(window, window + len(rolling)), rolling)
plt.xlabel("episode")
plt.ylabel(f"success rate (rolling mean over {window} episodes)")
plt.title("The agent learns to cross the ice")
plt.ylim(-0.05, 1.05)
plt.grid(alpha=0.3)
plt.show()

### 🧪 Exercise 5 — how good did it get?

Measure the agent's skill at the *end* of training: the mean success over the **last 200 episodes**. For a non-slippery 4×4 pond a trained agent should be essentially perfect.

In [ ]:
# ==================== YOUR CODE HERE ====================
### HINT: slice the last 200 with successes[-200:], then np.mean(...)
...
print(f"🎯 Success over the last 200 training episodes: {final_success:.1%}")
if not SMOKE:
    assert final_success >= 0.9, "Below 90% — double-check your Q-update line!"
    print("✅ The agent has essentially mastered the pond.")

## 5 · Look inside the brain

Those 64 numbers *are* the agent's mind. From them we can read out its complete strategy — a **policy**: in each state, the single best move.

### 🧪 Exercise 6 — read out the policy

Collapse each row of the table into two summaries:

- `best_action` — for every state, the action with the largest Q-value.
- `best_value` — for every state, how large that best value is.

Both are one call to a NumPy function along **axis 1** (across the 4 action-columns).

In [ ]:
# ==================== YOUR CODE HERE ====================
### HINT: axis=1 collapses the 4 action-columns, leaving one number per state
...
print("Best action per state (0=L 1=D 2=R 3=U), laid out on the 4x4 pond:")
print(best_action.reshape(4, 4))

In [ ]:
ARROWS = {0: "←", 1: "↓", 2: "→", 3: "↑"}   # ← ↓ → ↑
desc = env.unwrapped.desc.astype(str)      # 4x4 grid of letters

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Left: the learned policy as arrows
ax1.imshow(np.zeros((4, 4)), cmap="Blues", vmin=0, vmax=1)
for s in range(16):
    r, c = divmod(s, 4)
    tile = desc[r, c]
    if tile == "H":
        ax1.text(c, r, "H", ha="center", va="center", fontsize=18, color="red", weight="bold")
    elif tile == "G":
        ax1.text(c, r, "G", ha="center", va="center", fontsize=18, color="green", weight="bold")
    else:
        ax1.text(c, r, ARROWS[int(best_action[s])], ha="center", va="center", fontsize=26)
ax1.set_title("The learned policy (best move per tile)")
ax1.set_xticks([]); ax1.set_yticks([])

# Right: how valuable each tile is (max Q)
im = ax2.imshow(best_value.reshape(4, 4), cmap="viridis")
for s in range(16):
    r, c = divmod(s, 4)
    ax2.text(c, r, f"{best_value[s]:.2f}", ha="center", va="center",
             color="white", fontsize=10)
ax2.set_title("Value of each tile (max Q)")
ax2.set_xticks([]); ax2.set_yticks([])
fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()
print("64 numbers = a complete skater's instinct for this pond.")

## 6 · Watch it play 🎬

Numbers are convincing; a replay is delightful. We run **one greedy episode** (no exploration — pure exploitation of the table), capture every frame, and stitch them into a GIF.

In [ ]:
import imageio.v2 as imageio
from IPython.display import Image as IPImage, display

state, _ = env.reset(seed=SEED)
frames = [env.render()]
for _ in range(30):
    action = int(np.argmax(q[state]))          # greedy: always the best move
    state, reward, terminated, truncated, _ = env.step(action)
    frames.append(env.render())
    if terminated or truncated:
        break
env.close()

gif_path = REPO / "data" / "frozenlake_agent.gif"
imageio.mimsave(str(gif_path), frames, fps=3, loop=0)
print(f"Saved {len(frames)} frames → {gif_path}")
display(IPImage(filename=str(gif_path)))

## 7 · CartPole — when the cheat sheet explodes 🎪

The Q-table trick has a fatal limit. It works because FrozenLake has exactly **16 states** — you can list them in a table. Now meet **CartPole**: balance a pole upright by sliding a cart left or right. Its state is **four continuous numbers**:

`[cart position, cart velocity, pole angle, pole angular velocity]`

There are infinitely many possible states — you can't make a table with a row for every real number. So we **replace the table with a function that estimates Q-values**, and the best function approximator we know is a **neural network**. Feed it the 4 numbers, it outputs a Q-value for each action. That's a **Deep Q-Network (DQN)**.

Notice what just happened: **Day 5's neural networks have met today's rewards.** The course loops closed — the brains you built to *see* can also learn to *act*.

In [ ]:
env2 = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = env2.reset(seed=SEED)
print("A CartPole state is 4 continuous numbers:")
print("  [cart position, cart velocity, pole angle, pole angular velocity]")
print("  example:", np.round(obs, 3))
print("Actions: 0 = push cart LEFT, 1 = push cart RIGHT")
print("Reward: +1 for every step the pole stays up (500 = a perfect episode)")

### 🧪 Exercise 7 — the CartPole baseline

Same idea as Exercise 1: measure how a **random** agent does before we train anything. Run 20 episodes of random actions and report the **average number of steps** the pole stays up.

Expected outcome: only about **~20–25 steps** — a random cart drops the pole almost immediately.

In [ ]:
N_CARTPOLE = 20
baseline_env = gym.make("CartPole-v1")
baseline_env.action_space.seed(SEED)
lengths = []
for _ in range(N_CARTPOLE):
    obs, _ = baseline_env.reset()
    done, steps = False, 0
    # ==================== YOUR CODE HERE ====================
    ### HINT: count steps until the episode is terminated or truncated
    ...
    lengths.append(steps)
baseline_env.close()
print(f"🎲 Random agent balanced the pole for {np.mean(lengths):.1f} steps on average (500 = perfect)")

## 8 · Train a Deep Q-Network 🧠

Writing a solid DQN from scratch (replay buffers, target networks, the training loop) is a lot of careful code. Real labs don't reinvent it — they use a trusted library. We'll use **stable-baselines3** with a recipe tuned by the community for CartPole.

We train in **4 chunks** and evaluate after each one, so you can *watch* the mean reward climb from ~10 toward the maximum of 500. On a GPU the full run takes a few minutes; in smoke-test mode it's just a quick sanity pass.

In [ ]:
from stable_baselines3 import DQN

eval_env = gym.make("CartPole-v1")

def evaluate(model, n_episodes=5):
    """Average total reward over a few *greedy* (deterministic) episodes."""
    returns = []
    for _ in range(n_episodes):
        obs, _ = eval_env.reset()
        done, total = False, 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = eval_env.step(int(action))
            total += reward
            done = terminated or truncated
        returns.append(total)
    return float(np.mean(returns))

# Hyperparameters from the stable-baselines3 rl-zoo recipe for CartPole:
model = DQN("MlpPolicy", env2, learning_rate=2.3e-3, batch_size=64,
            buffer_size=100_000, learning_starts=1_000, gamma=0.99,
            target_update_interval=10, train_freq=256, gradient_steps=128,
            exploration_fraction=0.16, exploration_final_eps=0.04,
            policy_kwargs=dict(net_arch=[256, 256]), seed=SEED,
            device=DEVICE, verbose=0)

chunk = max(1, TIMESTEPS // 4)
print(f"Training the DQN for ~{TIMESTEPS} steps in 4 chunks (watch the reward climb)...")
for i in range(4):
    model.learn(total_timesteps=chunk, reset_num_timesteps=False, progress_bar=False)
    mean_reward = evaluate(model)
    print(f"  chunk {i+1}/4  (~{(i+1)*chunk:>6} steps)  mean reward = {mean_reward:6.1f}  (500 = perfect)")

print("🏁 DQN training complete!")

## 9 · Before vs. after — the wow moment ✨

The same task, two agents: an **untrained random** agent and **your trained DQN**. We record one episode of each as a GIF (capped at 500 steps, every 2nd frame to keep it small). Scroll between them — the difference is the whole point of today.

In [ ]:
import imageio.v2 as imageio
from IPython.display import Image as IPImage, display

def record_cartpole(model_or_none, path, seed=SEED, max_steps=500):
    """Run one CartPole episode, save a GIF, and return (path, steps survived)."""
    genv = gym.make("CartPole-v1", render_mode="rgb_array")
    genv.action_space.seed(seed)
    obs, _ = genv.reset(seed=seed)
    frames, steps = [], 0
    for step in range(max_steps):
        if model_or_none is None:
            action = genv.action_space.sample()               # random agent
        else:
            action, _ = model_or_none.predict(obs, deterministic=True)
            action = int(action)
        obs, reward, terminated, truncated, _ = genv.step(action)
        steps += 1
        if step % 2 == 0:                     # every 2nd frame keeps the GIF small
            frames.append(genv.render())
        if terminated or truncated:
            break
    genv.close()
    imageio.mimsave(str(path), frames, fps=25, loop=0)
    return path, steps

rand_path, rand_steps = record_cartpole(None,  REPO / "data" / "cartpole_random.gif")
dqn_path,  dqn_steps  = record_cartpole(model, REPO / "data" / "cartpole_trained.gif")

print(f"🎲 Random agent : survived {rand_steps} steps")
print(f"🧠 Trained DQN  : survived {dqn_steps} steps\n")
print("Random agent:")
display(IPImage(filename=str(rand_path)))
print("Your trained DQN:")
display(IPImage(filename=str(dqn_path)))

## 🔓 Unlock your Studio module

The 🎮 **Game Master** needs two files: the **Q-table** (`.npy`) and the **trained DQN** (`.zip`). Save both into your Studio's models folder to light up the tab.

In [ ]:
REQUIRED_FILES = ["game_master_qtable.npy", "game_master_dqn.zip"]

np.save(MODELS_DIR / "game_master_qtable.npy", q)
model.save(str(MODELS_DIR / "game_master_dqn.zip"))   # sb3 keeps the ".zip" name exactly as given

missing = [f for f in REQUIRED_FILES if not (MODELS_DIR / f).exists()]
assert not missing, f"Something didn't save: {missing}"

print("🔓 UNLOCKED: 🎮 Game Master!")
for f in REQUIRED_FILES:
    kb = (MODELS_DIR / f).stat().st_size / 1024
    print(f"   ✅ {f}  ({kb:.1f} KB)")
print("\nRun your Studio to play against both agents you trained:")
print("   python ai_studio/app.py   (from the repo root)")

## 🏁 Checkpoint

Today you crossed from **perceiving** to **acting**. You:

- Learned the **reinforcement-learning loop** — act, receive a reward and a new state, update your beliefs, repeat.
- Trained a **Q-table** on FrozenLake: 64 numbers that grew from all-zeros into a flawless instinct for the pond.
- Wrote the **Q-learning update** — the single line that turns raw experience into knowledge.
- Trained a **Deep Q-Network** on CartPole, where a neural network *replaces* the table — Day 5's brains meeting today's rewards.
- Unlocked the 🎮 **Game Master** in your Studio.

That's **seven modules built**. And here's the quiet truth about every one of them — the penguin classifier, the digit reader, the language models, today's agents: none were *programmed* with the answer. Each started knowing nothing and got better, from data or from rewards. Every capability in your Studio was **trained**.

**Tomorrow is Demo Day.** You'll assemble the entire AI Studio, rehearse your demo, and then run the **Grand Benchmark** — where we uncover the single mathematical operation humming underneath all seven modules, and discover why making it *fast* is the whole adventure of Weeks 3–4. See you there. 🚀